# Prework: Prepare 2022 Rainfall Event Data

This notebook prepares rainfall event data for Week 6.

## What this notebook does
1. Parse station metadata from `fungwong_202511.json`
2. Read selected 2022 daily rainfall CSV files
3. Merge rainfall records with station metadata by `station_id`
4. Export prepared outputs to CSV / JSON
5. Create Yilan + Hualien valid subsets for later interpolation work


## 0. User Settings

Please check the paths before running.


In [ ]:
from pathlib import Path
import json
import pandas as pd

# =========================
# User settings
# =========================
BASE_2022_DIR = Path(r"D:\Class_satellite\HW7\2022\2022")
STATION_JSON_PATH = Path(r"fungwong_202511.json")

# Candidate dates for Muifa + 1029 heavy rainfall event
TARGET_DATES = [
    "20220911", "20220912", "20220913", "20220914",
    "20221028", "20221029", "20221030", "20221031",
]

OUTPUT_DIR = Path("prework_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_COUNTIES = ["宜蘭縣", "花蓮縣"]

PRIMARY_RAIN_COL = "RAIN"
REMOVE_NEG998 = True
REMOVE_ZERO = True

print("BASE_2022_DIR =", BASE_2022_DIR)
print("STATION_JSON_PATH =", STATION_JSON_PATH)
print("OUTPUT_DIR =", OUTPUT_DIR.resolve())


## 1. Helper Functions


In [ ]:
def build_station_metadata(json_path: Path) -> pd.DataFrame:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    stations = data["records"]["Station"]
    rows = []

    for st in stations:
        coords = st.get("GeoInfo", {}).get("Coordinates", [])
        coord0 = coords[0] if coords else {}

        rows.append(
            {
                "station_id": st.get("StationId"),
                "station_name": st.get("StationName"),
                "county": st.get("GeoInfo", {}).get("CountyName"),
                "town": st.get("GeoInfo", {}).get("TownName"),
                "lat": coord0.get("StationLatitude"),
                "lon": coord0.get("StationLongitude"),
            }
        )

    meta = pd.DataFrame(rows).drop_duplicates(subset=["station_id"]).copy()
    return meta


def save_station_metadata(meta: pd.DataFrame, output_dir: Path) -> None:
    csv_path = output_dir / "station_metadata.csv"
    json_path = output_dir / "station_metadata.json"

    meta.to_csv(csv_path, index=False, encoding="utf-8-sig")
    meta.to_json(json_path, orient="records", force_ascii=False, indent=2)

    print(f"✅ Saved station metadata CSV: {csv_path}")
    print(f"✅ Saved station metadata JSON: {json_path}")


def find_rain_csv(base_dir: Path, date_str: str):
    matches = list(base_dir.rglob(f"rain_{date_str}.csv"))
    if not matches:
        return None
    if len(matches) > 1:
        print(f"⚠️ Multiple matches found for {date_str}, using first one:")
        for m in matches:
            print("   -", m)
    return matches[0]


def load_rain_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, encoding="utf-8")

    numeric_cols = ["ELEV", "RAIN", "MIN_10", "HOUR_3", "HOUR_6", "HOUR_12", "HOUR_24", "NOW"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "obsTime" in df.columns:
        df["obsTime"] = pd.to_datetime(df["obsTime"], errors="coerce")

    return df


def merge_rain_with_metadata(rain_df: pd.DataFrame, meta_df: pd.DataFrame) -> pd.DataFrame:
    return rain_df.merge(meta_df, on="station_id", how="left")


def filter_target_counties_and_valid_rain(
    merged_df: pd.DataFrame,
    counties,
    rain_col: str = "RAIN",
    remove_neg998: bool = True,
    remove_zero: bool = True,
) -> pd.DataFrame:
    out = merged_df.copy()

    if "county" in out.columns:
        out = out[out["county"].isin(counties)]

    if rain_col in out.columns:
        out = out.dropna(subset=[rain_col])
        if remove_neg998:
            out = out[out[rain_col] != -998]
        if remove_zero:
            out = out[out[rain_col] != 0]

    return out


def summarize_merge_quality(merged_df: pd.DataFrame) -> dict:
    total_rows = len(merged_df)
    total_station_ids = merged_df["station_id"].nunique() if "station_id" in merged_df.columns else None
    missing_meta_rows = merged_df["lat"].isna().sum() if "lat" in merged_df.columns else None
    missing_meta_ratio = (missing_meta_rows / total_rows) if total_rows and missing_meta_rows is not None else None

    return {
        "total_rows": total_rows,
        "unique_station_ids": total_station_ids,
        "missing_meta_rows": missing_meta_rows,
        "missing_meta_ratio": missing_meta_ratio,
    }


def export_prepared_outputs(
    all_merged: pd.DataFrame,
    county_valid: pd.DataFrame,
    date_str: str,
    output_dir: Path,
) -> None:
    date_dir = output_dir / date_str
    date_dir.mkdir(exist_ok=True)

    merged_csv = date_dir / f"rain_{date_str}_merged.csv"
    merged_json = date_dir / f"rain_{date_str}_merged.json"

    county_csv = date_dir / f"rain_{date_str}_yl_hl_valid.csv"
    county_json = date_dir / f"rain_{date_str}_yl_hl_valid.json"

    all_merged.to_csv(merged_csv, index=False, encoding="utf-8-sig")
    all_merged.to_json(merged_json, orient="records", force_ascii=False, indent=2, date_format="iso")

    county_valid.to_csv(county_csv, index=False, encoding="utf-8-sig")
    county_valid.to_json(county_json, orient="records", force_ascii=False, indent=2, date_format="iso")

    print(f"✅ Saved merged CSV: {merged_csv}")
    print(f"✅ Saved merged JSON: {merged_json}")
    print(f"✅ Saved Yilan/Hualien valid CSV: {county_csv}")
    print(f"✅ Saved Yilan/Hualien valid JSON: {county_json}")


## 2. Build and Save Station Metadata


In [ ]:
station_meta = build_station_metadata(STATION_JSON_PATH)
print("station_meta shape:", station_meta.shape)
display(station_meta.head())

save_station_metadata(station_meta, OUTPUT_DIR)


## 3. Process All Target Dates


In [ ]:
summary_rows = []

for date_str in TARGET_DATES:
    print("\n" + "=" * 72)
    print(f"Processing {date_str} ...")

    csv_path = find_rain_csv(BASE_2022_DIR, date_str)
    if csv_path is None:
        print(f"❌ CSV not found for {date_str}")
        summary_rows.append(
            {
                "date": date_str,
                "status": "missing_csv",
                "csv_path": None,
                "total_rows": None,
                "unique_station_ids": None,
                "missing_meta_rows": None,
                "missing_meta_ratio": None,
                "yl_hl_rows": None,
                "yl_hl_station_count": None,
            }
        )
        continue

    print(f"📂 CSV: {csv_path}")

    rain_df = load_rain_csv(csv_path)
    merged_df = merge_rain_with_metadata(rain_df, station_meta)

    quality = summarize_merge_quality(merged_df)

    yl_hl_valid = filter_target_counties_and_valid_rain(
        merged_df,
        counties=TARGET_COUNTIES,
        rain_col=PRIMARY_RAIN_COL,
        remove_neg998=REMOVE_NEG998,
        remove_zero=REMOVE_ZERO,
    )

    export_prepared_outputs(merged_df, yl_hl_valid, date_str, OUTPUT_DIR)

    yl_hl_station_count = yl_hl_valid["station_id"].nunique() if len(yl_hl_valid) else 0

    summary_rows.append(
        {
            "date": date_str,
            "status": "ok",
            "csv_path": str(csv_path),
            "total_rows": quality["total_rows"],
            "unique_station_ids": quality["unique_station_ids"],
            "missing_meta_rows": quality["missing_meta_rows"],
            "missing_meta_ratio": quality["missing_meta_ratio"],
            "yl_hl_rows": len(yl_hl_valid),
            "yl_hl_station_count": yl_hl_station_count,
            "yl_hl_rain_mean": yl_hl_valid[PRIMARY_RAIN_COL].mean() if len(yl_hl_valid) else None,
            "yl_hl_rain_max": yl_hl_valid[PRIMARY_RAIN_COL].max() if len(yl_hl_valid) else None,
            "yl_hl_rain_std": yl_hl_valid[PRIMARY_RAIN_COL].std() if len(yl_hl_valid) else None,
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


## 4. Save Summary


In [ ]:
summary_csv = OUTPUT_DIR / "prework_summary.csv"
summary_json = OUTPUT_DIR / "prework_summary.json"

summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
summary_df.to_json(summary_json, orient="records", force_ascii=False, indent=2)

print(f"✅ Saved summary CSV: {summary_csv}")
print(f"✅ Saved summary JSON: {summary_json}")


## 5. Quick Preview of One Prepared File

You can change the date string below if needed.


In [ ]:
preview_date = "20220912"

preview_csv = OUTPUT_DIR / preview_date / f"rain_{preview_date}_yl_hl_valid.csv"
if preview_csv.exists():
    preview_df = pd.read_csv(preview_csv, encoding="utf-8")
    print(preview_df.shape)
    display(preview_df.head())
else:
    print("Preview file not found:", preview_csv)


## 6. Notes

After this notebook finishes, you will have:

1. `station_metadata.csv` / `station_metadata.json`
2. one folder per date with merged CSV / JSON
3. Yilan + Hualien valid subset CSV / JSON
4. `prework_summary.csv` / `prework_summary.json`

These outputs are ready for the next Week 6 analysis step.
